# KArSL Arabic Sign Language Recognition — Modular Pipeline

This notebook is a **thin orchestration layer**.  
All logic lives in the Python modules; cells here simply call them.

| Module | Responsibility |
|---|---|
| `config/config.py` | All paths, hyperparameters, experiment definitions |
| `data/extract.py` | Download, 7z extraction, video relocation |
| `data/loader.py` | Load `.npy` files, train/val/test splits, save arrays |
| `features/keypoints.py` | MediaPipe initialisation, frame processing, normalisation |
| `features/augmentation.py` | Full video → `.npy` pipeline across all signers/classes |
| `model/architecture.py` | ArSL-TGRU model definition |
| `training/trainer.py` | Compile, callbacks, training loop |
| `evaluation/evaluate.py` | Metrics, per-class report, summary table |

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q gdown py7zr
!pip install -q protobuf==5.29.1
!pip install -q mediapipe==0.10.21 --no-deps

import sys, os
# Add the project root to the Python path so all modules resolve correctly
PROJECT_ROOT = '/content/KArSL_modular'   # ← adjust if you cloned elsewhere
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Setup done ✓')

## 1 · Download & Extract Dataset

In [ ]:
from data.extract import extract_dataset, fix_misplaced_videos, inspect_dataset, NOTEBOOK_CORRECTIONS

# Download dataset (run once)
# from data.extract import download_dataset
# download_dataset('https://drive.google.com/drive/folders/1R8ZI6a1xgRNqUThDFA-X4RT7FJh9WeRr',
#                  '/content/drive/MyDrive/arabic_dataset/')

extract_dataset()
fix_misplaced_videos(NOTEBOOK_CORRECTIONS)
inspect_dataset()

## 2 · Keypoint Extraction + Augmentation

In [ ]:
from features.augmentation import run_augmentation_pipeline

run_augmentation_pipeline()

## 3 · Prepare Experiment Splits

In [ ]:
from data.loader import prepare_all_experiments

prepare_all_experiments()

## 4 · Train — Experiment 1 (Test signer 03)

In [ ]:
from data.loader import load_experiment
from training.trainer import train_experiment

data = load_experiment('Exp1_Test_03')
model_exp1, history_exp1 = train_experiment(
    exp_name='Exp1_Test_03',
    X_train=data['X_train'], y_train=data['y_train'],
    X_val=data['X_val'],   y_val=data['y_val'],
)

## 4b · Evaluate Experiment 1

In [ ]:
from evaluation.evaluate import evaluate_model

evaluate_model(model_exp1, data['X_test'], data['y_test'], exp_name='Exp1 — Unseen Signer 03')

## 5 · Train — Experiment 2 (Test signer 02)

In [ ]:
data = load_experiment('Exp2_Test_02')
model_exp2, history_exp2 = train_experiment(
    exp_name='Exp2_Test_02',
    X_train=data['X_train'], y_train=data['y_train'],
    X_val=data['X_val'],   y_val=data['y_val'],
)
evaluate_model(model_exp2, data['X_test'], data['y_test'], exp_name='Exp2 — Unseen Signer 02')

## 6 · Train — Experiment 3 (Test signer 01)

In [ ]:
data = load_experiment('Exp3_Test_01')
model_exp3, history_exp3 = train_experiment(
    exp_name='Exp3_Test_01',
    X_train=data['X_train'], y_train=data['y_train'],
    X_val=data['X_val'],   y_val=data['y_val'],
)
evaluate_model(model_exp3, data['X_test'], data['y_test'], exp_name='Exp3 — Unseen Signer 01')

## 7 · Summary Table

In [ ]:
from evaluation.evaluate import evaluate_model, summarise_all_experiments
from data.loader import load_experiment

experiments = [
    ('Exp1_Test_03', model_exp1, 'Exp1 — Test 03'),
    ('Exp2_Test_02', model_exp2, 'Exp2 — Test 02'),
    ('Exp3_Test_01', model_exp3, 'Exp3 — Test 01'),
]

all_results = []
for exp_name, model, label in experiments:
    d = load_experiment(exp_name)
    m = evaluate_model(model, d['X_test'], d['y_test'], exp_name=label)
    m['name'] = label
    all_results.append(m)

summarise_all_experiments(all_results)